In [1]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def relu(self):
        # Forward pass
        out_data = self.data if self.data > 0 else 0.0
        out = Value(out_data, (self,), 'ReLU')

        def _backward():
            local_grad = 1.0 if out.data > 0 else 0.0
            self.grad += local_grad * out.grad

        out._backward = _backward
        return out

    def backward(self):
        # Topological order of computation graph
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)

        # Backpropagation
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


# --- Problem 2: Building the Neuron ---
print("--- Micro-Autograd Neuron ---")

# Inputs
x1 = Value(2.0)
x2 = Value(-3.0)

# Weights
w1 = Value(-1.0)
w2 = Value(1.0)

# Bias
b = Value(6.0)

# Forward pass
w1x1 = w1 * x1
w2x2 = w2 * x2
dot = w1x1 + w2x2
neuron = dot + b
out = neuron.relu()

print(f"Forward Pass Output: {out.data}")

# Backward pass
out.backward()

print("\n--- Gradients ---")
print(f"Gradient w1 (d_out/d_w1): {w1.grad}")
print(f"Gradient w2 (d_out/d_w2): {w2.grad}")
print(f"Gradient x1 (d_out/d_x1): {x1.grad}")
print(f"Gradient x2 (d_out/d_x2): {x2.grad}")
print(f"Gradient b (d_out/d_b): {b.grad}")

--- Micro-Autograd Neuron ---
Forward Pass Output: 1.0

--- Gradients ---
Gradient w1 (d_out/d_w1): 2.0
Gradient w2 (d_out/d_w2): -3.0
Gradient x1 (d_out/d_x1): -1.0
Gradient x2 (d_out/d_x2): 1.0
Gradient b (d_out/d_b): 1.0
